In [0]:
import os
import sys
import numpy as np
import rasterio
import geopandas as gpd
import pandas as pd
from skimage.filters import threshold_otsu
from rasterstats import zonal_stats
from concurrent.futures import ProcessPoolExecutor, as_completed
import warnings
from functools import reduce
from operator import mul
from rasterio.features import geometry_mask
from rasterio.warp import reproject, Resampling
from shapely.geometry import mapping
import re
from rasterio.mask import mask
from rasterio.io import MemoryFile
import tools
from pathlib import Path
import pipelines
import mosaicVI
import traceback
import glob

In [0]:
# ── Unity Catalog location (must match the other scripts in this pipeline) ──
# See the README's "Key concepts" section for what catalog/schema mean.
CATALOG = "use1_prod_artemis_catalog_3718194974443840"  # Change
SCHEMA = "tier1_raw"  # Change

# Read the full flight inventory table generated by Job 1 (1_Flights_update).
TABLE_NAME = f"{CATALOG}.{SCHEMA}.drone_mission_table"
df_misiones = spark.table(TABLE_NAME)

# Collect all flights into a local Python list so we can loop through them
# one by one (this pipeline processes flights individually, not in bulk
# Spark operations).
flights = df_misiones.collect()

total_vuelos = len(flights)

# Counters for the final summary report.
processed = 0
skipped = 0
errors = 0

# ── Go through every flight and compute vegetation indices for it ──────────
for i, vuelo in enumerate(flights, 1):

    print(f"\n[{i}/{total_vuelos}] Checking inventory for mission: {vuelo['mission']} | Lote: {vuelo['field']}...")

    # Work out this flight's folder structure:
    # flight_metadata_path -> date_folder -> mission_folder
    fd = vuelo["flight_metadata_path"]
    date_folder = os.path.dirname(fd)
    mission_folder = os.path.dirname(date_folder)

    # The plot boundary geometry lives inside 'field_data'. We don't assume a
    # fixed filename: we pick the first .geojson (or .shp) found in that folder,
    # mirroring the plot-cropping script's flexible lookup.
    field_data_dir = os.path.join(mission_folder, "field_data")

    vector_files = (
        glob.glob(os.path.join(field_data_dir, "*.geojson"))
        + glob.glob(os.path.join(field_data_dir, "*.shp"))
    )

    # Where the resulting index CSV will be saved for this mission.
    output_csv = f"{mission_folder}/ouput_INDEX.csv"

    # Skip this flight if its index CSV already exists — this makes the
    # script safe to re-run: it only processes flights that haven't been
    # done yet.
    if not os.path.exists(output_csv):
        try:
            # ----------------------------------------------------------
            # Existence check ONLY.
            # The updated pipeline library descends into the date folders on
            # its own and finds the orthomosaic under the new nested layout
            #   <date_folder>/orthomosaico/agisoft_YYYY_MM_DD/SENSOR/RGB.tif
            # so we must pass it `mission_folder` (the folder that CONTAINS
            # the date folders), NOT the sensor folder. We still run a quick
            # glob here just to skip flights whose orthomosaic isn't ready.
            # ----------------------------------------------------------
            possible_ortho_names = ["RGB.tif", "MS.tif"]
            ortho_found = any(
                glob.glob(f"{date_folder}/*/*/*/{name}")
                for name in possible_ortho_names
            )

            # DEBUG: verify what we found before calling the pipeline.
            print(f"   [DEBUG] mission_folder     = {mission_folder}")
            print(f"   [DEBUG] date_folder        = {date_folder}")
            print(f"   [DEBUG] ortho_found        = {ortho_found}")
            print(f"   [DEBUG] field_data_dir     = {field_data_dir}")
            print(f"   [DEBUG] vector_files       = {vector_files}")

            # If no orthomosaic exists yet, skip this flight cleanly.
            if not ortho_found:
                print(f" Skipped. No RGB.tif or MS.tif found under {date_folder}.")
                skipped += 1
                continue

            # If no plot boundary geometry exists, skip this flight cleanly.
            if not vector_files:
                print(f" Skipped. No .geojson/.shp found in {field_data_dir}.")
                skipped += 1
                continue

            polygon_file_path = vector_files[0]

            # Check which type of imagery this flight has, so we know which
            # processing pipeline to use (same detection logic used in the
            # Agisoft processing script).
            raw_data_dir = os.path.join(date_folder, "raw_data")
            multi_spec_dir = os.path.join(raw_data_dir, "multi-spec")

            if os.path.exists(multi_spec_dir):
                # Multispectral flight -> use the 4-band index pipeline
                # (e.g. NDVI and other vegetation indices that require
                # multiple spectral bands).
                results = pipelines.fourband()

                results.process_multispectral_data(
                    mission_folder,     # pipeline descends into <date>/.../MS.tif
                    polygon_file_path,
                    output_csv,
                    r"",                # (unused/placeholder argument)
                    tools
                )
            else:
                # Standard RGB flight -> use the RGB-only index pipeline
                # (visible-light-based indices instead of NDVI).
                results = pipelines.RGB()

                results.process_RGB_data(
                    mission_folder,     # pipeline descends into <date>/.../RGB.tif
                    polygon_file_path,
                    output_csv,
                    r"",                # (unused/placeholder argument)
                    tools
                )

            print(f" CSV generated successfully in: {output_csv}")
            processed += 1

        except Exception as e:
            # Log the error, but keep going with the next flight rather than
            # stopping the whole batch. Print the full traceback so the exact
            # failing line inside the pipeline is visible.
            print(f" Error processing mission {vuelo['mission']}: {e}")
            traceback.print_exc()
            errors += 1

    else:
        print(f" Skipped. CSV already exists.")
        skipped += 1

# ── Final summary report ────────────────────────────────────────────────────
print("\n" + "="*45)
print("BATCH RGB PROCESSING RESULTS")
print("="*45)
print(f"▶ New cases: {processed}")
print(f"▶ Skipped (already existed): {skipped}")
print(f"▶ Execution errors: {errors}")

Total flights recorded in the table: 7

[1/7] Reviewing DEM inventory for mission: mission_idk | Lote: bhavanisagar...
 ⏭ Skipped. CSV already exists.

[2/7] Reviewing DEM inventory for mission: mission_100 | Lote: field_100...
 ⏭ Skipped. CSV already exists.

[3/7] Reviewing DEM inventory for mission: prueba_3 | Lote: field_06...
 ⏭ Skipped. CSV already exists.

[4/7] Reviewing DEM inventory for mission: corn_health_assessment_flight | Lote: west_field_03...
 ⏭ Skipped. CSV already exists.

[5/7] Reviewing DEM inventory for mission: prueba_2 | Lote: west_field_03...
 ⏭ Skipped. CSV already exists.

[6/7] Reviewing DEM inventory for mission: prueba_4 | Lote: field_10...
 ⏭ Skipped. CSV already exists.

[7/7] Reviewing DEM inventory for mission: prueba_5 | Lote: field_11...
 ⏭ Skipped. CSV already exists.

BATCH DEM PROCESSING RESULTS
▶ New cases: 0
▶ Skipped (already existed): 7
▶ Execution errors: 0


In [0]:
# ── Unity Catalog location (must match the other scripts in this pipeline) ──
# See the README's "Key concepts" section for what catalog/schema mean.
CATALOG = "use1_prod_artemis_catalog_3718194974443840" #Change
SCHEMA = "tier1_raw" #Change
 
# Read the full flight inventory table generated by Job 1 (1_Flights_update).
TABLE_NAME = f"{CATALOG}.{SCHEMA}.drone_mission_table"
df_misiones = spark.table(TABLE_NAME)
 
# Collect all flights into a local Python list so we can loop through them
# one by one (this pipeline processes flights individually, not in bulk
# Spark operations).
flights = df_misiones.collect()
 
total_vuelos = len(flights)
 
# Counters for the final summary report.
processed = 0
skipped = 0
errors = 0
 
# ── Go through every flight and compute vegetation indices for it ──────────
for i, vuelo in enumerate(flights, 1):
 
    print(f"\n[{i}/{total_vuelos}] Checking inventory for mission: {vuelo['mission']} | Lote: {vuelo['field']}...")
 
    # Work out this flight's folder structure:
    # flight_metadata_path -> date_folder -> mission_folder
    fd = vuelo["flight_metadata_path"]
    date_folder = os.path.dirname(fd)
    mission_folder = os.path.dirname(date_folder)
 
    # The plot boundary geometry used to compute per-plot index values.
    field_data_dir = os.path.join(mission_folder, "field_data")
    polygon_file_path = f"{field_data_dir}/plot_boundary.geojson"
 
    # Where the resulting index CSV will be saved for this mission.
    output_csv = f"{mission_folder}/ouput_INDEX.csv"
 
    # Skip this flight if its index CSV already exists — this makes the
    # script safe to re-run: it only processes flights that haven't been
    # done yet.
    if not os.path.exists(output_csv):
        try:
            # Check which type of imagery this flight has, so we know which
            # processing pipeline to use (same detection logic used in the
            # Agisoft processing script).
            raw_data_dir = os.path.join(date_folder, "raw_data")
            multi_spec_dir = os.path.join(raw_data_dir, "multi-spec")
 
            if os.path.exists(multi_spec_dir):
                # Multispectral flight -> use the 4-band index pipeline
                # (e.g. NDVI and other vegetation indices that require
                # multiple spectral bands).
                results = pipelines.fourband()
 
                results.process_multispectral_data(
                    mission_folder,
                    polygon_file_path,
                    output_csv,
                    r"",   # (unused/placeholder argument)
                    tools
                )
            else:
                # Standard RGB flight -> use the RGB-only index pipeline
                # (visible-light-based indices instead of NDVI).
                results = pipelines.RGB()
 
                results.process_RGB_data(
                    mission_folder,
                    polygon_file_path,
                    output_csv,
                    r"",   # (unused/placeholder argument)
                    tools
                )
 
            print(f" CSV generated successfully in: {output_csv}")
            processed += 1
 
        except Exception as e:
            # Log the error, but keep going with the next flight rather than
            # stopping the whole batch.
            print(f" Error processing mission {vuelo['mission']}: {e}")
            errors += 1
 
    else:
        print(f" Skipped. CSV already exists.")
        skipped += 1
 
# ── Final summary report ────────────────────────────────────────────────────
print("\n" + "="*45)
print("BATCH RGB PROCESSING RESULTS")
print("="*45)
print(f"▶ New cases: {processed}")
print(f"▶ Skipped (already existed): {skipped}")
print(f"▶ Execution errors: {errors}")


[1/7] Checking inventory for mission: mission_idk | Lote: bhavanisagar...
Processing VIS for folder: 2025-02-25
Shapefile loaded and reprojected
Calculating Otsu
THE ORIGINAL THRESHOLD IS 0.4331617057323456
Processing canopy metrics for folder: 2025-02-25
CRS: EPSG:4326  |  Pixel: 0.0094 m × 0.0095 m  |  Area: 0.000089 m²
Combined results exported successfully to /Volumes/use1_prod_artemis_catalog_3718194974443840/production/data/pheno_google/CIAT_CALI/bhavanisagar/best/2025:ind:soybean:unknown/bhavanisagar/section_a/drone/mission_idk/ouput_INDEX.csv
 CSV generated successfully in: /Volumes/use1_prod_artemis_catalog_3718194974443840/production/data/pheno_google/CIAT_CALI/bhavanisagar/best/2025:ind:soybean:unknown/bhavanisagar/section_a/drone/mission_idk/ouput_INDEX.csv

[2/7] Checking inventory for mission: mission_100 | Lote: field_100...
 Skipped. CSV already exists.

[3/7] Checking inventory for mission: prueba_3 | Lote: field_06...
 Skipped. CSV already exists.

[4/7] Checking inv